## 📊 Entraînement d'un modèle CNN sur des images de radiographies pulmonaires

### 📁 Chargement des bibliothèques et définition des arguments
Importation des bibliothèques nécessaires et configuration des paramètres d'entraînement.

In [6]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
import json
from datetime import datetime
import time
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

### 📂 Préparation des Données pour l'Entraînement d'un Modèle de Classification d'Images
Ce bloc de code est dédié à la préparation des données pour l'entraînement d'un modèle de classification d'images, en utilisant des radiographies pulmonaires. Les images sont chargées depuis les répertoires ``train`` et ``val``, et une augmentation des données est appliquée pour améliorer la robustesse du modèle.

Étapes Clés :
1. Définition des Répertoires : Les chemins vers les répertoires de données d'entraînement et de validation sont spécifiés.
2. Paramètres Globaux : Configuration des paramètres tels que la taille des images, la taille des lots (batch size), et le nombre d'époques pour l'entraînement.
3. Augmentation des Données : Utilisation de ``ImageDataGenerator`` pour appliquer diverses transformations aux images d'entraînement, telles que la rotation, le décalage, le zoom, et le flip horizontal.
4. Chargement des Images : Les images sont chargées et préparées pour l'entraînement et la validation, avec redimensionnement et normalisation des pixels.

In [7]:
cwd = os.getcwd()

# Data directories, follow this structure
train_dir = os.path.join(cwd, 'chest_xray', 'train')
validation_dir = os.path.join(cwd, 'chest_xray', 'val')

# Paramètres globaux de traitement des images
IMG_SIZE = (224, 224)  # Taille des images après redimensionnement
BATCH_SIZE = 32        # Taille du batch pour l'entraînement
EPOCHS = 15            # Nombre epochs d'entraînement

# Générateur de données pour l'entraînement avec augmentation d'images
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalisation des pixels entre 0 et 1
    rotation_range=20,        # Rotation aléatoire des images jusqu'à 20°
    width_shift_range=0.2,    # Décalage horizontal des images jusqu'à 20%
    height_shift_range=0.2,   # Décalage vertical des images jusqu'à 20%
    zoom_range=0.2,           # Zoom aléatoire jusqu'à 20%
    horizontal_flip=True      # Flip horizontal aléatoire
)

# Générateur de données pour la validation (sans augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

# Chargement des images d'entraînement à partir du répertoire
train_generator = train_datagen.flow_from_directory(
    train_dir, 
    target_size=IMG_SIZE,     # Redimensionnement des images
    batch_size=BATCH_SIZE,    # Taille du batch
    class_mode='binary'       # Classification binaire (pneumonie ou non)
)

# Chargement des images de validation à partir du répertoire
val_generator = val_datagen.flow_from_directory(
    validation_dir, 
    target_size=IMG_SIZE,     # Redimensionnement des images
    batch_size=BATCH_SIZE,    # Taille du batch
    class_mode='binary'       # Classification binaire (pneumonie ou non)
)

Found 5216 images belonging to 2 classes.
Found 36 images belonging to 2 classes.


### 🧠 Construction du Modèle CNN pour la Classification d'Images
Ce bloc de code décrit la construction d'un réseau de neurones convolutionnel (CNN) destiné à la classification d'images, en particulier pour la détection de la pneumonie à partir de radiographies pulmonaires. Le modèle est conçu avec plusieurs couches de convolution, des activations ReLU, des couches de pooling, et des mécanismes de régularisation comme le dropout pour améliorer les performances et éviter le surapprentissage.

Architecture du Modèle :
1. Couches de Convolution : Trois couches de convolution avec des filtres de taille croissante (32, 64, 128) pour extraire des caractéristiques de plus en plus complexes des images.
2. Activation ReLU : Utilisée après chaque couche de convolution pour introduire la non-linéarité.
3. Pooling : Couches de MaxPooling pour réduire la dimension spatiale des cartes de caractéristiques, tout en conservant les informations importantes.
4. Aplatissement : Conversion des cartes de caractéristiques 2D en un vecteur 1D pour les couches denses.
5. Couches Denses : Une couche dense avec 128 neurones, suivie d'une couche de sortie avec une activation softmax pour la classification binaire (Normal vs Pneumonie).
6. Dropout : Technique de régularisation pour éviter le surapprentissage en désactivant aléatoirement 50% des neurones pendant l'entraînement.

In [8]:
# Définition du modèle de réseau de neurones convolutionnel (CNN)
model = models.Sequential([
    # Première couche de convolution avec 32 filtres de taille 3x3 et activation ReLU
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D((2,2)),  # Couche de pooling pour réduire la dimension spatiale

    # Deuxième couche de convolution avec 64 filtres
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Troisième couche de convolution avec 128 filtres
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Aplatissement des cartes de caractéristiques
    layers.Flatten(),
    
    # Couche dense entièrement connectée avec 128 neurones et activation ReLU
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Dropout de 50% pour éviter le surapprentissage
    
    # Couche de sortie avec une activation sigmoïde pour une classification binaire
    layers.Dense(1, activation='sigmoid')  # Sortie binaire (0 = NORMAL, 1 = PNEUMONIA)
])

/home/atressel/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### ⚙️ Compilation et Résumé du Modèle CNN
Ce bloc de code se concentre sur la compilation du modèle CNN et l'affichage de son résumé. La compilation configure les aspects essentiels de l'entraînement, tels que l'optimiseur, la fonction de perte, et les métriques d'évaluation. Le résumé du modèle fournit une vue d'ensemble de son architecture, y compris les couches et le nombre de paramètres.

Configuration de la Compilation :
1. Optimiseur : Utilisation de l'optimiseur Adam, qui ajuste automatiquement le taux d'apprentissage pour chaque paramètre, ce qui permet une convergence plus rapide et efficace.
2. Fonction de Perte : binary_crossentropy, adaptée à une classification binaire, mesure l'erreur entre les prédictions du modèle et les valeurs réelles.
3. Métriques :
    * Accuracy : Précision globale du modèle sur les prédictions.
    * Categorical Accuracy : Exactitude des prédictions pour chaque catégorie.
    * Precision : Précision des prédictions positives, indiquant combien de prédictions positives sont correctes.
    * Recall : Capacité du modèle à identifier tous les cas positifs.
    * AUC : Aire sous la courbe ROC, un indicateur global de la performance du modèle en termes de discrimination entre les classes.

In [9]:
# Compilation du modèle avec l'optimiseur Adam et la fonction de perte adaptée à une classification binaire
model.compile(
    optimizer='adam',  # Optimiseur adaptatif qui ajuste le taux d'apprentissage automatiquement
    loss='binary_crossentropy',  # Fonction de perte pour la classification binaire
    metrics=[  # Liste des métriques utilisées pour évaluer la performance du modèle
        'accuracy',  # Précision globale des prédictions
        'categorical_accuracy',  # Exactitude pour chaque catégorie
        'precision',  # Précision des prédictions positives
        'recall',  # Rappel des prédictions positives
        'auc'  # Aire sous la courbe ROC (indicateur global de performance)
    ]
)

# Affichage du résumé du modèle pour visualiser les couches et le nombre de paramètres
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

### 🏋️‍♂️  Entraînement du Modèle CNN avec Gestion du Déséquilibre de Classes
Ce bloc de code se concentre sur l'entraînement du modèle CNN en utilisant la méthode ``model.fit()``. Il inclut également une gestion du déséquilibre de classes en calculant des poids spécifiques pour chaque classe, afin d'améliorer la performance du modèle sur les classes sous-représentées.

Étapes de l'Entraînement :
1. Calcul des Poids de Classe :
    * Utilisation de compute_class_weight pour déterminer les poids des classes, ce qui permet de compenser le déséquilibre en attribuant plus d'importance aux classes minoritaires.
    * Création d'un dictionnaire class_weight_dict pour associer chaque classe à son poids calculé.
2. Entraînement du Modèle :
    * Données d'Entraînement : Utilisation de train_generator pour fournir les images d'entraînement avec augmentation des données.
    * Données de Validation : Utilisation de val_generator pour évaluer la performance du modèle sur des données non vues pendant l'entraînement.
    * Nombre d'Époques : Le modèle est entraîné sur un nombre d'époques défini précédemment (EPOCHS).
    * Poids de Classe : Application des poids de classe pour équilibrer l'importance des différentes classes pendant l'entraînement.
3. Enregistrement du Temps :
    * Le temps de début de l'entraînement est enregistré pour permettre le suivi de la durée totale de l'entraînement.

In [10]:
# Enregistrement du temps de début de l'entraînement
start_time = time.time()

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

# Entraînement du modèle sur les données d'entraînement et validation sur les données de validation
history = model.fit(
    train_generator,  # Données d'entraînement générées
    validation_data=val_generator,  # Données de validation
    epochs=EPOCHS,  # Nombre epochs définies précédemment
    class_weight=class_weight_dict,  # Poids de classe pour gérer le déséquilibre
)

/home/atressel/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 56s 324ms/step - accuracy: 0.6760 - auc: 0.7252 - categorical_accuracy: 0.7727 - loss: 0.7841 - precision: 0.8479 - recall: 0.6864 - val_accuracy: 0.5278 - val_auc: 0.6295 - val_categorical_accuracy: 0.1111 - val_loss: 0.9344 - val_precision: 0.7895 - val_recall: 0.5357
Epoch 2/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 45s 276ms/step - accuracy: 0.8312 - auc: 0.9152 - categorical_accuracy: 0.7408 - loss: 0.3670 - precision: 0.9601 - recall: 0.8056 - val_accuracy: 0.5833 - val_auc: 0.6585 - val_categorical_accuracy: 0.1111 - val_loss: 0.8242 - val_precision: 0.8095 - val_recall: 0.6071
Epoch 3/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 45s 275ms/step - accuracy: 0.8514 - auc: 0.9208 - categorical_accuracy: 0.7816 - loss: 0.3566 - precision: 0.9600 - recall: 0.8344 - val_accuracy: 0.7222 - val_auc: 0.7478 - val_categorical_accuracy: 1.0000 - val_loss: 0.8292 - val_precision: 0.8750 - val_recall: 0.7500
Epoch 4/15
163/163 ━━━━━━━━━━━━━━━━━━━━ 45s 276ms/step - accuracy: 

## ⏱️ Évaluation de la Durée d'Entraînement et Sauvegarde du Modèle CNN
Ce bloc de code traite de l'évaluation de la durée d'entraînement du modèle CNN et de la sauvegarde du modèle entraîné. Il inclut également la récupération et la préparation des métriques d'entraînement et de validation pour une analyse ultérieure.

Étapes Clés :
1. Calcul de la Durée d'Entraînement :
    * Mesure du temps écoulé entre le début et la fin de l'entraînement pour évaluer la durée totale.
    * Conversion de cette durée en un format lisible (heures, minutes, secondes) pour une meilleure compréhension.
2. Récupération des Métriques :
    * Extraction des métriques d'entraînement et de validation à partir de l'objet history, incluant la précision, la perte, l'AUC, la précision catégorielle, la précision des prédictions positives, et le rappel.
    * Ces métriques permettent d'évaluer la performance du modèle au fil des époques.
3. Sauvegarde du Modèle :
    * Création d'un répertoire models pour stocker les modèles entraînés.
    * Génération d'un nom de fichier unique basé sur la date et l'heure actuelles pour éviter les conflits de noms.
    * Sauvegarde du modèle au format HDF5 pour une utilisation future ou une évaluation supplémentaire.
4. Préparation des Données pour l'Analyse :
    * Définition de la plage des époques utilisées pour l'entraînement, ce qui facilite le traçage des métriques.

In [11]:

# Calcul de la durée totale de l'entraînement
end_time = time.time()
training_duration = end_time - start_time  # Durée en secondes

# Récupération des métriques d'entraînement et de validation au fil des epochs
acc = history.history['accuracy'] # Précision sur l'ensemble d'entraînement
auc = history.history['auc'] # Aire sous la courbe ROC
categorical_accuracy = history.history['categorical_accuracy'] # Exactitude pour chaque catégorie
loss = history.history['loss'] # Perte sur l'ensemble d'entraînement
precision = history.history['precision'] # Précision des prédictions positives
recall = history.history['recall'] # Rappel des prédictions positives
val_accuracy = history.history['val_accuracy'] # Précision sur l'ensemble de validation
val_auc = history.history['val_auc'] # Aire sous la courbe ROC sur l'ensemble de validation
val_categorical_accuracy = history.history['val_categorical_accuracy'] # Exactitude pour chaque catégorie sur l'ensemble de validation
val_loss = history.history['val_loss'] # Perte sur l'ensemble de validation
val_precision = history.history['val_precision'] # Précision des prédictions positives sur l'ensemble de validation
val_recall = history.history['val_recall'] # Rappel des prédictions positives sur l'ensemble de validation

# Définition de la plage des epochs utilisées pour l'entraînement
epochs_range = range(EPOCHS)

# Création du répertoire de sauvegarde du modèle s'il n'existe pas
os.makedirs("models", exist_ok=True)

# Génération d'un nom de fichier unique basé sur la date et l'heure
date = datetime.now().strftime('%Y%m%d%H%M%S')
model_save_path = f"models/pneumonia-{date}.h5"

# Sauvegarde du modèle entraîné au format HDF5
model.save(model_save_path)

# Enregistrement du temps d'entraînement dans un format lisible
log_id = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
hours, remainder = divmod(training_duration, 3600)
minutes, seconds = divmod(remainder, 60)
formatted_time = f"{int(hours)} hours, {int(minutes)} minutes, {seconds:.2f} seconds"


### 📝 Journalisation de l'Entraînement du Modèle CNN
Ce bloc de code se concentre sur la création d'un fichier de log JSON détaillé, qui enregistre toutes les métriques d'entraînement et de validation du modèle CNN. Ce fichier de log est essentiel pour suivre les performances du modèle et pour une analyse approfondie ultérieure.

Contenu du Fichier de Log :
1. Informations Clés de l'Entraînement :
    * Chemin du Modèle : Emplacement où le modèle entraîné est sauvegardé.
    * Métriques Finales : Précision, perte, précision des prédictions positives, rappel, AUC, et score F1 pour les ensembles d'entraînement et de validation.
    * Paramètres d'Entraînement : Nombre d'époques, taille du batch, et taille des images utilisées.
    * Durée d'Entraînement : Temps total écoulé pour l'entraînement, formaté pour une meilleure lisibilité.
2. Informations sur les Données :
    * Nombre d'Échantillons : Quantité d'échantillons dans les ensembles d'entraînement et de validation.
    * Distribution des Classes : Informations sur la répartition des classes dans les données d'entraînement.
3. Métriques par Époque :
    * Enregistrement des métriques de précision, perte, précision des prédictions positives, rappel, et AUC pour chaque époque, à la fois pour l'entraînement et la validation.
    * Métriques formatées pour une présentation claire et concise.
4. Chemin du Graphique :
    * Emplacement du graphique généré qui visualise les métriques d'entraînement et de validation.

In [12]:
# Création d'une entrée de log contenant les informations clés de l'entraînement
log_entry = {
    log_id: {
        "model_path": model_save_path,  # Chemin du modèle sauvegardé
        "accuracy": acc[-1],  # Dernière valeur de la précision sur l'ensemble d'entraînement
        "loss": loss[-1],  # Dernière valeur de la perte sur l'ensemble d'entraînement
        "accuracy_validation": val_accuracy[-1],  # Dernière précision sur l'ensemble de validation
        "loss_validation": val_loss[-1],  # Dernière perte sur l'ensemble de validation
        "precision": precision[-1], # Précision des prédictions positives
        "recall": recall[-1], # Rappel des prédictions positives
        "auc": auc[-1], # Aire sous la courbe ROC
        "f1_score": 2 * (precision[-1] * recall[-1]) / (precision[-1] + recall[-1]) if precision[-1] + recall[-1] > 0 else 0,
        "epochs": EPOCHS,  # Nombre total d'époques d'entraînement
        "batch_size": BATCH_SIZE,  # Taille du batch utilisé
        "img_size": IMG_SIZE,  # Taille des images utilisées
        "training_duration": formatted_time,  # Durée totale de l'entraînement
        "graph_path": f"reports/graphs/accuracy-loss-{date}.png",  # Chemin du graphique généré
        "dataset_info": { # Informations sur les données d'entraînement et de validation
            "train_samples": train_generator.samples, # Nombre d'échantillons dans l'ensemble d'entraînement
            "val_samples": val_generator.samples, # Nombre d'échantillons dans l'ensemble de validation
            "class_distribution": train_generator.class_indices # Distribution des classes
        },
        "metrics_per_epoch": { # Métriques par epoch pour l'entraînement et la validation
            "accuracy": acc, # Précision par epoch sur l'ensemble d'entraînement
            "val_accuracy": val_accuracy, # Précision par epoch sur l'ensemble de validation
            "loss": loss, # Perte par epoch sur l'ensemble d'entraînement
            "val_loss": val_loss, # Perte par epoch sur l'ensemble de validation
            "precision": precision, # Précision par epoch des prédictions positives
            "recall": recall, # Rappel par epoch des prédictions positives
            "auc": auc # Aire sous la courbe ROC par epoch
        },
        "formatted_metrics": { # Métriques formatées pour affichage
            "accuracy": [f"{a:.2%}" for a in acc], # Précision formatée par epoch
            "val_accuracy": [f"{a:.2%}" for a in val_accuracy], # Précision de validation formatée par epoch
            "loss": [f"{l:.4f}" for l in loss], # Perte formatée par epoch
            "val_loss": [f"{l:.4f}" for l in val_loss], # Perte de validation formatée par epoch
            "precision": [f"{p:.2%}" for p in precision], # Précision des prédictions positives formatée par epoch
            "recall": [f"{r:.2%}" for r in recall], # Rappel des prédictions positives formaté par epoch
            "auc": [f"{a:.4f}" for a in auc] # Aire sous la courbe ROC formatée par epoch
        }
    }
}

### 📈 Visualisation des Performances du Modèle CNN
Ce bloc de code se concentre sur la visualisation des performances du modèle CNN au fil des époques d'entraînement. Il affiche les courbes de précision et de perte pour les ensembles d'entraînement et de validation, permettant ainsi une évaluation visuelle du comportement et de la convergence du modèle.

Étapes de Visualisation :
1. Gestion des Logs :
    * Chargement des logs existants à partir d'un fichier JSON pour conserver un historique des entraînements.
    * Mise à jour du fichier de log avec les nouvelles métriques d'entraînement.
2. Création des Graphiques :
    * Précision du Modèle : Affichage des courbes de précision pour les ensembles d'entraînement et de validation, permettant de comparer la performance du modèle sur les données vues et non vues.
    * Perte du Modèle : Affichage des courbes de perte pour les ensembles d'entraînement et de validation, aidant à identifier le surapprentissage ou le sous-apprentissage.
3. Sauvegarde des Graphiques :
    * Les graphiques sont sauvegardés sous forme d'images PNG dans un répertoire dédié, facilitant leur consultation ultérieure.
    * Utilisation de ``plt.close()`` pour éviter l'affichage multiple des figures lors de l'exécution dans un environnement interactif.
4. Configuration des Graphiques :
    * Utilisation de matplotlib pour créer des figures avec une taille définie et des sous-figures pour une présentation claire des métriques.
    * Ajout de légendes, d'étiquettes d'axes, et de titres pour améliorer la lisibilité et la compréhension des graphiques.

In [13]:
# Définition du chemin du fichier de log
log_file_path = "reports/training_log.json"

# Création du répertoire s'il n'existe pas
os.makedirs("reports", exist_ok=True)

# Chargement des logs existants s'ils sont déjà présents
if os.path.exists(log_file_path):
    with open(log_file_path, "r") as f:
        existing_logs = json.load(f)
else:
    existing_logs = {}

# Mise à jour du fichier de log avec la nouvelle entrée
existing_logs.update(log_entry)

# Écriture des logs mis à jour dans le fichier JSON
with open(log_file_path, "w") as f:
    json.dump(existing_logs, f, indent=4)  # Indentation pour une meilleure lisibilité

# Création d'une figure pour afficher les courbes d'apprentissage
plt.figure(figsize=(12, 5))  # Définition de la taille de la figure

# Première sous-figure : Précision du modèle
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', color='tab:blue')  # Précision sur l'entraînement
plt.plot(epochs_range, val_accuracy, label='Validation Accuracy', color='tab:red')  # Précision sur la validation
plt.xlabel('Epochs')  # Étiquette de l'axe des abscisses
plt.ylabel('Accuracy')  # Étiquette de l'axe des ordonnées
plt.title('Model Accuracy')  # Titre du graphique
plt.legend()  # Affichage de la légende

# Deuxième sous-figure : Perte du modèle
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', color='tab:blue')  # Perte sur l'entraînement
plt.plot(epochs_range, val_loss, label='Validation Loss', color='tab:red')  # Perte sur la validation
plt.xlabel('Epochs')  # Étiquette de l'axe des abscisses
plt.ylabel('Loss')  # Étiquette de l'axe des ordonnées
plt.title('Model Loss')  # Titre du graphique
plt.legend(loc=1)  # Affichage de la légende en haut à droite

# # Précision
# plt.subplot(1, 2, 1)
# plt.plot(epochs_range, acc, label='Training Accuracy')
# plt.plot(epochs_range, val_accuracy, label='Validation Accuracy')
# plt.legend(loc='lower right')
# plt.title('Accuracy')

# # Perte
# plt.subplot(1, 2, 2)
# plt.plot(epochs_range, loss, label='Training Loss')
# plt.plot(epochs_range, val_loss, label='Validation Loss')
# plt.legend(loc='upper right')
# plt.title('Loss')

# Création du répertoire s'il n'existe pas
os.makedirs("reports/graphs", exist_ok=True)

# Sauvegarde du graphique sous forme d'image PNG
plt.savefig(f"reports/graphs/accuracy-loss-{date}.png")

# Fermeture de la figure pour éviter l'affichage multiple
plt.close()